In [ ]:
! pip install rasterio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import rasterio
import numpy as np 
from matplotlib import pyplot as plt 
from matplotlib.colors import ListedColormap
import cv2
from matplotlib import cm
import pandas as pd 
import geopandas as gpd 
from rasterio.plot import show 
import seaborn as sns


In [ ]:
# Path to image
path = r('.tiff')

In [ ]:
src  = rasterio.open(path)
im = src.read()

print(im.shape) # Channel, Row and Column


In [ ]:
# RGB PLOT
R = im[:, :, 2]
G = im[:, :, 1]
B = im[:, :, 0]


In [ ]:
rgb = np.dstack((R,G, B))

In [ ]:
plt.figure(figsize=(16, 12))
plt.show(rgb*4)
plt.axis('off')

In [ ]:
# Training Samples
samples = gpd.readfile('.shp')
samples

In [ ]:
print(samples.crs)


In [ ]:
# New column to add class names
samples['label'] = samples['id'].replace({0:'Water', 1:'Bare', 2:'Vegetation', 3:'BuiltUp'})
samples

In [ ]:
cmap = ListedColormap(['blue', 'brown', 'green', 'red'])
fig, ax = plt.subplots(figsize = (20, 20))
samples.plot(column = 'label', categorical = True, cmap = cmap, legend  = True, legend_kwds = {'bbox_to_anchor':})
ax.axis('off')
show(rgb.transpose([2,0, 1])*4, transform= src.transform, ax = ax)

In [ ]:
# Value Extraction
samples['geometry']

In [ ]:
array_samples = []
for point in samples['geometry']:
    x = point.xy[0][0]
    y = point.xy[1][0]

    row, col = src.index(x, y)

    band_values = []
    for i in range(src.count):
        band_values.append(src.read(i+1)[row, col])
    array_samples.append(band_values)

In [ ]:
# List to array
X = np.array(array_samples)
X.shape

In [ ]:
dataset = pd.DataFrame(data=X, columns=['B1', 'B2','B3'])
dataset['label'] = samples['id']
dataset

In [ ]:
# Separating Spectral Data from the target
X = dataset.iloc[:, 0:-1].values
Y = dataset.iloc[:, -1].values


In [ ]:
# Sklearn Function Importation
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import OneHotEncoder
from sklearn.neural_network import MLPClassifier


In [ ]:
new_y = Y[:, np.newaxis]
new_y.shape

In [ ]:
enc = OneHotEncoder()
enc.fit(new_y)
oneHotLabels = enc.transform(new_y).toarray()

In [ ]:
oneHotLabels.shape

In [ ]:
oneHotLabels

In [ ]:
# Training Testing
x_train, x_test, y_train, y_test = train_test_split(X, oneHotLabels, test_size=0.2, random_state=42)

In [ ]:
classifier = MLPClassifier(
    hidden_layer_sizes=(16, 32, 8), 
    max_iter=5000, 
    activation='relu',
    verbose=10
)

In [ ]:
classifier.fit(x_train,y_train)

In [ ]:
# Prediction
y_pred = classifier.predict(x_test)

In [ ]:
# Accuracy
accuracy_score(y_test, y_pred)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
# Predicting usage and Coverage map using the trained model
with rasterio.open(path) as src:
    im = src.read()
    

In [ ]:
out_meta = src.meta.copy()

In [ ]:
im = im.transpose([1,2,0])
X = np.nan_to_num(im)


In [ ]:
flatten_x = X.reshape(X.shape[0] * X.shape[1], X.shape[2])

In [ ]:
pred = classifier.predict(flatten_x)
pred

In [ ]:
classify = np.argmax(pred, axis=1)
classify = classify + 1
classify = classify.reshape(X.shape[0], X.shape[1])

In [ ]:
plt.figure(figsize=(16, 10))
plt.imshow(classify, cmap = 'tab20c')
plt.axis('off')

In [ ]:
export_image = classify[np.newaxis, :, :]
out_meta.update({
    'drivers':'GTiff',
    'height':export_image.shape[1],
    'width': export_image[2],
    'compress':'lzw',
    'count':1
})

In [ ]:
with rasterio.open('/content/lulc.tiff', "w",
                   **out_meta) as dest:
    dest.write(export_image)